In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
PATH = '/content/drive'

from google.colab import drive
drive.mount(PATH)

In [ ]:
DATASET_PATH = '/content/drive/My Drive/recs'

In [ ]:
LABEL_MAP = {'ur': 0, 'en': 1, 'mix': 2}
REVERSE_LABEL_MAP = {0: 'Urdu', 1: 'English', 2: 'Mixed'}

In [ ]:
def extract_features(file_path):
    """
    Extracts 13-dim MFCCs from an audio file and adds a bias term.
    [cite_start]Ref: [cite: 126] (13-dim MFCC)[cite_start], [cite: 129] (Bias x0=1)[cite_start], [cite: 103] (16kHz)
    """
    try:
        # Load audio with 16000 Hz sampling rate
        audio, sample_rate = librosa.load(file_path, sr=16000)

        # Extract MFCCs (returns shape: 13, Time)
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=13)

        # Take the MEAN of MFCCs over time to get a single vector per audio file
        mfccs_processed = np.mean(mfccs.T, axis=0)

        # Add Bias term (x0 = 1) at the beginning
        feature_vector = np.insert(mfccs_processed, 0, 1)

        return feature_vector
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [ ]:
def load_data(folder_path):
    X = []
    y = []

    print("Loading data and extracting features...")
    files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]

    for file_name in files:
        # Extract label from filename (e.g., "ur-1-123.wav" -> "ur")
        # [cite_start]Ref: [cite: 117] Naming convention
        try:
            lang_code = file_name.split('-')[0]
            if lang_code not in LABEL_MAP:
                continue # Skip if naming format is wrong

            label = LABEL_MAP[lang_code]
            file_path = os.path.join(folder_path, file_name)

            features = extract_features(file_path)

            if features is not None:
                X.append(features)
                y.append(label)
        except IndexError:
            continue

    return np.array(X), np.array(y)

In [ ]:
X, y = load_data(DATASET_PATH)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data Loaded. Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

In [ ]:
class MultinomialLogisticRegression:
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.loss_history = []

    def softmax(self, z):
        # [cite_start]Ref: [cite: 135] Softmax function
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True)) # Stability fix
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def one_hot_encode(self, y, num_classes):
        return np.eye(num_classes)[y]

    def cross_entropy_loss(self, y_true_one_hot, y_prob):
        # [cite_start]Ref: [cite: 136] Cross-entropy loss
        # Add epsilon to prevent log(0)
        epsilon = 1e-15
        y_prob = np.clip(y_prob, epsilon, 1 - epsilon)
        loss = -np.mean(np.sum(y_true_one_hot * np.log(y_prob), axis=1))
        return loss

    def fit(self, X, y):
        num_samples, num_features = X.shape
        num_classes = len(np.unique(y))

        # Initialize weights (Features x Classes)
        self.weights = np.zeros((num_features, num_classes))
        y_one_hot = self.one_hot_encode(y, num_classes)

        # [cite_start]Batch Gradient Descent [cite: 137]
        for epoch in range(self.epochs):
            # 1. Linear model: Z = X * W
            z = np.dot(X, self.weights)

            # 2. Activation: Softmax
            y_pred = self.softmax(z)

            # 3. Calculate Loss (for plotting)
            loss = self.cross_entropy_loss(y_one_hot, y_pred)
            self.loss_history.append(loss)

            # 4. Gradient Calculation
            gradient = np.dot(X.T, (y_pred - y_one_hot)) / num_samples

            # 5. Update Weights
            self.weights -= self.lr * gradient

            if epoch % 100 == 0:
                print(f"Epoch {epoch}: Loss {loss:.4f}")

    def predict(self, X):
        # [cite_start]Ref: [cite: 138] Prediction function
        z = np.dot(X, self.weights)
        y_probs = self.softmax(z)
        return np.argmax(y_probs, axis=1)

In [ ]:
print("\nTraining Model from Scratch...")
scratch_model = MultinomialLogisticRegression(learning_rate=0.01, epochs=1500)
scratch_model.fit(X_train, y_train)

# [cite_start]Plot Loss Curve [cite: 140]
plt.plot(scratch_model.loss_history)
plt.title('Training Loss vs Iterations')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.show()

In [ ]:
def evaluate_performance(y_true, y_pred):
    # [cite_start]Ref: [cite: 139] Evaluation function (Accuracy, Macro-Precision/Recall/F1, Confusion Matrix)

    # 1. Confusion Matrix
    num_classes = 3
    conf_matrix = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        conf_matrix[t][p] += 1

    # 2. Accuracy
    accuracy = np.trace(conf_matrix) / np.sum(conf_matrix)

    # 3. Macro Average Metrics
    precision_sum = 0
    recall_sum = 0
    f1_sum = 0

    for i in range(num_classes):
        tp = conf_matrix[i][i]
        fp = np.sum(conf_matrix[:, i]) - tp
        fn = np.sum(conf_matrix[i, :]) - tp

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        precision_sum += precision
        recall_sum += recall
        f1_sum += f1

    macro_precision = precision_sum / num_classes
    macro_recall = recall_sum / num_classes
    macro_f1 = f1_sum / num_classes

    print("\n--- Evaluation (Scratch Model) ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro Precision: {macro_precision:.4f}")
    print(f"Macro Recall: {macro_recall:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    print("Confusion Matrix:\n", conf_matrix)

In [ ]:
y_pred_scratch = scratch_model.predict(X_test)
evaluate_performance(y_test, y_pred_scratch)

In [ ]:
print("\n--- Scikit-Learn Implementation ---")

In [ ]:
sklearn_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=2000)
sklearn_model.fit(X_train, y_train)

In [ ]:
y_pred_sklearn = sklearn_model.predict(X_test)

In [ ]:
sk_acc = accuracy_score(y_test, y_pred_sklearn)


In [ ]:
sk_cm = confusion_matrix(y_test, y_pred_sklearn)

In [ ]:
print(f"Sklearn Accuracy: {sk_acc:.4f}")
print("Sklearn Confusion Matrix:\n", sk_cm)

In [ ]:
def predict_single_file(file_path, model):
    """
    Takes a path to a .wav file, extracts features, and predicts the language.
    """
    print(f"Processing: {file_path}")

    features = extract_features(file_path)

    if features is None:
        return "Error reading file."


    features_reshaped = features.reshape(1, -1)

    prediction_index = model.predict(features_reshaped)[0]

    predicted_label = REVERSE_LABEL_MAP[prediction_index]

    return predicted_label

In [ ]:
# --- USAGE ---

# Replace 'test_voice.wav' with the actual name of the file you uploaded
new_file_path = '/content/drive/My Drive/test_mix.wav'

# Check if file exists before running
if os.path.exists(new_file_path):
    # Test with your Scratch Model
    result_scratch = predict_single_file(new_file_path, scratch_model)
    print(f"Scratch Model predicts: {result_scratch}")

    # Test with Scikit-Learn Model
    result_sklearn = predict_single_file(new_file_path, sklearn_model)
    print(f"Sklearn Model predicts: {result_sklearn}")
else:
    print("File not found! Please upload a .wav file to Colab files sidebar.")